In [1]:
from transformers import AutoTokenizer, AutoModel

model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [2]:
import torch
from torch import nn

# MPS runs tensor operations on the Apple GPU. Keep float32 for stable training.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training device: {device}")


Training device: mps


In [3]:
from __future__ import annotations

import json
from typing import Any

import torch


QTYPES = {
    "choice": 0,
    "score": 1,
    "noul": 2,
}


def prepare_laya_inputs(
    schema: dict[str, Any],
    tokenizer,
    max_len: int = 512,
    head_max_len: int = 192,
):
    """
    Convert a Laya-style typed request into batched model inputs.

    Expected schema:
    {
        "state": str | dict | list,
        "questions": {
            "question_id": {
                "type": "choice" | "score" | "noul",
                "instructions": str,
                "criteria": ...
            },
            ...
        }
    }

    Returns:
        {
            "input_ids":      LongTensor [n_questions, seq_len],
            "attention_mask": LongTensor [n_questions, seq_len],
            "marker_pos":     LongTensor [n_questions, max_options],
            "marker_mask":    BoolTensor [n_questions, max_options],
            "qtype":          LongTensor [n_questions],
            "question_ids":   list[str],
        }
    """

    state = schema["state"]
    questions = schema["questions"]

    mask_token = tokenizer.mask_token
    mask_id = tokenizer.mask_token_id
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id

    # ------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------

    def encode(text: str, **kwargs) -> list[int]:
        return tokenizer(
            text,
            add_special_tokens=False,
            **kwargs,
        )["input_ids"]

    def serialize_state(value: Any) -> str:
        if isinstance(value, str):
            return value
        return json.dumps(value, ensure_ascii=False)

    def render_criterion(value: Any) -> str:
        if isinstance(value, str):
            return value

        return json.dumps(
            value,
            ensure_ascii=False,
            separators=(", ", ": "),
            default=str,
        )

    def normalize_question(q: dict[str, Any]) -> dict[str, Any]:
        """
        Equivalent in spirit to Laya's Agent._to_internal().
        """

        qtype = q["type"]
        criteria = q.get("criteria")

        # choice can be:
        #
        # ["billing", "technical"]
        #
        # instead of:
        #
        # {
        #   "billing": "...",
        #   "technical": "..."
        # }
        if qtype == "choice" and isinstance(criteria, list):
            criteria = {
                label: None
                for label in criteria
            }

        # noul can contain bool keys.
        if qtype == "noul" and isinstance(criteria, dict):
            criteria = {
                str(k).lower(): v
                for k, v in criteria.items()
            }

        instructions = q["instructions"]

        if not isinstance(instructions, str):
            instructions = json.dumps(
                instructions,
                ensure_ascii=False,
            )

        internal = {
            "t": qtype,
            "ins": instructions,
            "crit": criteria,
        }

        if "labels" in q:
            internal["labels"] = q["labels"]

        if "option_order" in q:
            internal["option_order"] = q["option_order"]

        return internal

    def render_options(q: dict[str, Any]) -> list[str]:
        qtype = q["t"]
        criteria = q.get("crit")

        # -------------------------
        # choice
        # -------------------------
        if qtype == "choice":
            return [
                str(label)
                if description is None or description == ""
                else f"{label}: {render_criterion(description)}"
                for label, description in criteria.items()
            ]

        # -------------------------
        # score
        # -------------------------
        if qtype == "score":
            return [
                f"level {i}: {render_criterion(description)}"
                for i, description in enumerate(criteria)
            ]

        # -------------------------
        # noul
        # -------------------------
        criteria = criteria or {}

        labels = q.get(
            "labels",
            {
                "false": "false",
                "true": "true",
            },
        )

        false_description = criteria.get("false")
        true_description = criteria.get("true")

        if false_description in (None, ""):
            false_description = "no, the statement does not hold"
        else:
            false_description = render_criterion(false_description)

        if true_description in (None, ""):
            true_description = "yes, the statement holds"
        else:
            true_description = render_criterion(true_description)

        return [
            f"{labels['false']}: {false_description}",
            f"{labels['true']}: {true_description}",
        ]

    # ------------------------------------------------------------
    # State is tokenized ONCE and reused for every question.
    # ------------------------------------------------------------

    state_text = serialize_state(state).replace(mask_token, " ")

    state_ids = encode(state_text)

    # Conversation-list states preserve the most recent tokens.
    truncate_left = isinstance(state, list)

    items = []
    question_ids = []

    # ------------------------------------------------------------
    # One model sequence per question.
    # ------------------------------------------------------------

    for question_id, public_q in questions.items():
        q = normalize_question(public_q)

        options = render_options(q)

        order = q.get(
            "option_order",
            list(range(len(options))),
        )

        instructions = q["ins"].replace(mask_token, " ")

        # Example:
        #
        # "choice question: Which department should handle this?"
        question_text = (
            f"{q['t']} question: {instructions}"
        )

        head_ids = encode(question_text)

        # --------------------------------------------------------
        # Each answer option becomes:
        #
        # [MASK] option text...
        # --------------------------------------------------------

        option_ids = []

        for option_index in order:
            option_text = (
                " "
                + options[option_index].replace(mask_token, " ")
            )

            tokens = encode(
                option_text,
                truncation=True,
                max_length=48,
            )

            option_ids.append(
                [mask_id] + tokens
            )

        # --------------------------------------------------------
        # Laya's head budget.
        # --------------------------------------------------------

        option_budget = (
            head_max_len
            - sum(len(o) for o in option_ids)
        )

        if option_budget < 16:
            per_option = max(
                4,
                (head_max_len - 16)
                // max(1, len(option_ids)),
            )

            option_ids = [
                option[:per_option]
                for option in option_ids
            ]

            option_budget = (
                head_max_len
                - sum(len(o) for o in option_ids)
            )

        head_ids = head_ids[
            : max(8, option_budget)
        ]

        # --------------------------------------------------------
        # Construct:
        #
        # [CLS]
        # question
        # [SEP]
        # [MASK] option 0
        # [MASK] option 1
        # ...
        # [SEP]
        # state
        # [SEP]
        # --------------------------------------------------------

        ids = (
            [cls_id]
            + head_ids
            + [sep_id]
        )

        markers = []

        for option in option_ids:
            # Position of the [MASK] token.
            markers.append(len(ids))

            ids.extend(option)

        ids.append(sep_id)

        # Leave one token for final [SEP].
        state_room = max(
            0,
            max_len - len(ids) - 1,
        )

        if truncate_left:
            # Conversation:
            # preserve newest tokens.
            selected_state_ids = (
                state_ids[-state_room:]
                if state_room > 0
                else []
            )
        else:
            selected_state_ids = (
                state_ids[:state_room]
            )

        ids = (
            ids
            + selected_state_ids
            + [sep_id]
        )

        ids = ids[:max_len]

        # Drop any marker that somehow fell outside max_len.
        markers = [
            marker
            for marker in markers
            if marker < max_len
        ]

        items.append(
            {
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
            }
        )

        question_ids.append(question_id)

    # ------------------------------------------------------------
    # Batch questions
    # ------------------------------------------------------------

    batch_size = len(items)

    if batch_size == 0:
        raise ValueError("At least one question is required.")

    seq_len = max(
        len(item["ids"])
        for item in items
    )

    max_options = max(
        len(item["markers"])
        for item in items
    )

    input_ids = torch.full(
        (batch_size, seq_len),
        pad_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (batch_size, seq_len),
        dtype=torch.long,
    )

    marker_pos = torch.zeros(
        (batch_size, max_options),
        dtype=torch.long,
    )

    marker_mask = torch.zeros(
        (batch_size, max_options),
        dtype=torch.bool,
    )

    qtype = torch.tensor(
        [
            item["qtype"]
            for item in items
        ],
        dtype=torch.long,
    )

    for row, item in enumerate(items):
        length = len(item["ids"])

        input_ids[row, :length] = torch.tensor(
            item["ids"],
            dtype=torch.long,
        )

        attention_mask[row, :length] = 1

        num_options = len(item["markers"])

        marker_pos[row, :num_options] = torch.tensor(
            item["markers"],
            dtype=torch.long,
        )

        marker_mask[row, :num_options] = True

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "marker_pos": marker_pos,
        "marker_mask": marker_mask,
        "qtype": qtype,

        # Useful for mapping model rows back to questions.
        "question_ids": question_ids,
    }

In [4]:
class OptionScorer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.layers = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, 1)
        )

    def forward(self, inputs, mask, temp=1.0):
        logits = (self.layers(inputs)/temp).squeeze(-1)
        logits = logits.masked_fill(torch.logical_not(mask), -1e4) 
        return logits
        

In [5]:
class DecisionModel(nn.Module):
    def __init__(self, d_model, nhead, num_layers, dim_feedforward, dropout):
        super().__init__()
        # Load only the encoder and use PyTorch attention supported on MPS.
        self.bert = AutoModel.from_pretrained(
            model_id, attn_implementation="sdpa", dtype=torch.float32
        )
        # BERT is a fixed feature extractor; only the decision layers train.
        self.bert.requires_grad_(False)
        self.bert.eval()
        if d_model != self.bert.config.hidden_size:
            raise ValueError("d_model must match the encoder hidden size")
        self.question_type = nn.Embedding(3, d_model)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, batch_first=True,
                dim_feedforward=dim_feedforward, dropout=dropout,
            ),
            num_layers=num_layers,
            enable_nested_tensor=False,
        )
        self.option_scorer = OptionScorer(d_model)

    def train(self, mode=True):
        super().train(mode)
        # Keep the frozen encoder deterministic when the head enters train mode.
        self.bert.eval()
        return self

    def forward(self, typed_input):
        inputs = typed_input["input_ids"]
        # no_grad avoids retaining encoder activations for backward.
        # These ordinary tensors can still be used by the trainable head.
        with torch.no_grad():
            outputs = self.bert(
                input_ids=inputs, attention_mask=typed_input["attention_mask"]
            ).last_hidden_state
        q_embed = self.question_type(typed_input["qtype"])
        decision_input = q_embed.unsqueeze(1) + outputs
        transformer_output = self.transformer(
            decision_input,
            src_key_padding_mask=typed_input["attention_mask"] == 0,
        )
        # Gather markers without creating an index tensor on the CPU.
        marker_indices = typed_input["marker_pos"].unsqueeze(-1).expand(
            -1, -1, transformer_output.shape[-1]
        )
        masks_only = transformer_output.gather(1, marker_indices)
        return self.option_scorer(masks_only, typed_input["marker_mask"])


In [6]:
dec = DecisionModel(768, 16, 2, 4096, 0.1).to(device)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def reward(output, target, qtype, marker_mask):
    #print(output, target)
    s_log = torch.sum(target * torch.log(torch.clamp(output, min=1e-12)), dim=-1).mean(dim=-1)
    s_sph = (torch.sum(target * output,dim=-1)/torch.sqrt(torch.sum(output**2,dim=-1))).mean(dim=-1)
    s_rps = (((qtype == 1)*(1/(marker_mask.sum(dim=-1))*torch.sum((output.cumsum(dim=-1)-target.cumsum(dim=-1))**2,dim=-1))).sum(dim=-1))
    #print(s_log, s_sph, s_rps)
    return s_log + 0.5*s_sph - s_rps

In [8]:
from datasets import load_dataset

ds = load_dataset("LocalLLaMA/typed-decisions", "customer_service")

In [9]:
def ds_to_schema(ds, i):
    # Read a row once instead of repeatedly accessing whole dataset columns.
    row = ds[i]
    return {
        "state": json.loads(row["state"]),
        "questions": json.loads(row["questions"]),
    }


In [10]:
def ds_to_target(ds, i, inputs):
    row = ds[i]
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])

    mask = inputs["marker_mask"]
    target = torch.zeros(
        mask.shape, dtype=torch.float32, device=mask.device
    )

    for row, question_id in enumerate(inputs["question_ids"]):
        question = questions[question_id]
        qtype = question["type"]

        # Canonical option identities, matching prepare_laya_inputs().
        if qtype == "choice":
            criteria = question["criteria"]
            labels = (
                list(criteria.keys())
                if isinstance(criteria, dict)
                else list(criteria)
            )
            labels = [str(label) for label in labels]

        elif qtype == "score":
            labels = [
                str(level)
                for level in range(len(question["criteria"]))
            ]

        elif qtype == "noul":
            # Display labels do not change false/true semantics.
            labels = ["false", "true"]

        else:
            raise ValueError(f"Unsupported question type: {qtype}")

        if not labels or len(set(labels)) != len(labels):
            raise ValueError(
                f"{question_id}: options must be nonempty and unique"
            )

        order = list(
            question.get("option_order", range(len(labels)))
        )
        if sorted(order) != list(range(len(labels))):
            raise ValueError(
                f"{question_id}: option_order must be a permutation"
            )

        presented_labels = [labels[index] for index in order]

        if int(mask[row].sum().item()) != len(presented_labels):
            raise ValueError(
                f"{question_id}: input markers do not match options; "
                "check whether preprocessing dropped any options"
            )

        probabilities = gold[question_id]["probabilities"]
        values = torch.tensor(
            [probabilities[label] for label in presented_labels],
            dtype=target.dtype,
            device=target.device,
        )

        if (
            not torch.isfinite(values).all()
            or (values < 0).any()
            or not torch.isclose(
                values.sum(), values.new_tensor(1.0),
                atol=1e-4, rtol=0,
            )
        ):
            raise ValueError(
                f"{question_id}: invalid target probabilities"
            )

        target[row, mask[row]] = values

    return target


def prepare_dataset_batch(ds, indices, tokenizer):
    """Flatten real questions from dataset rows, padding tokens and options on CPU.

    row_spans contains (question_start, question_end, original_option_width)
    for each dataset row, preserving its reward and loss reductions.
    """
    examples = []
    targets = []
    row_spans = []
    question_start = 0
    for index in indices:
        inputs = prepare_laya_inputs(ds_to_schema(ds, index), tokenizer)
        targets.append(ds_to_target(ds, index, inputs))
        examples.append(inputs)
        num_questions, num_options = inputs["marker_pos"].shape
        row_spans.append((question_start, question_start + num_questions, num_options))
        question_start += num_questions

    if not examples:
        raise ValueError("At least one dataset row is required")

    seq_len = max(inputs["input_ids"].shape[1] for inputs in examples)
    max_options = max(inputs["marker_pos"].shape[1] for inputs in examples)

    def pad_and_cat(key, width, value=0):
        return torch.cat([
            torch.nn.functional.pad(
                inputs[key], (0, width - inputs[key].shape[1]), value=value
            )
            for inputs in examples
        ], dim=0)

    inputs = {
        "input_ids": pad_and_cat("input_ids", seq_len, tokenizer.pad_token_id),
        "attention_mask": pad_and_cat("attention_mask", seq_len),
        "marker_pos": pad_and_cat("marker_pos", max_options),
        "marker_mask": pad_and_cat("marker_mask", max_options, False),
        "qtype": torch.cat([item["qtype"] for item in examples]),
        "question_ids": [qid for item in examples for qid in item["question_ids"]],
    }
    target = torch.cat([
        torch.nn.functional.pad(item, (0, max_options - item.shape[1]))
        for item in targets
    ], dim=0)
    return inputs, target, row_spans


def batch_candidate_loss(candidates, target, qtype, marker_mask, row_spans, sigma):
    """Average independent row losses; normalize advantages within each row."""
    from torch.distributions.normal import Normal

    with torch.no_grad():
        probabilities = torch.softmax(candidates, dim=-1)
        row_rewards = torch.stack([
            reward(
                probabilities[:, start:end, :width],
                target[start:end, :width].unsqueeze(0),
                qtype[start:end],
                marker_mask[start:end, :width],
            )
            for start, end, width in row_spans
        ], dim=1)  # [G, number of dataset rows]
        advantage = (
            row_rewards - row_rewards.mean(dim=0, keepdim=True)
        ) / (row_rewards.std(dim=0, keepdim=True) + 1e-6)

    # Keep the existing zero-centered Normal objective for each dataset row.
    normal = Normal(
        candidates.new_zeros(()), candidates.new_tensor(sigma), validate_args=False
    )
    log_prob = normal.log_prob(candidates).masked_fill(
        ~marker_mask.unsqueeze(0), 0.0
    )
    # Each row has equal weight regardless of its question or option count.
    return torch.stack([
        -(advantage[:, row].view(-1, 1, 1) * log_prob[:, start:end, :width]).mean()
        for row, (start, end, width) in enumerate(row_spans)
    ]).mean()



In [11]:
sigma = 1.0
G = 8
batch_size = 16  # Dataset rows per optimizer step; each row can contain many questions.
log_every = 1
max_steps = None  # Optimizer steps (batches); None processes all dataset rows.


In [12]:
if sigma <= 0 or G < 2:
    raise ValueError("sigma must be positive and G must be at least 2")
if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
    raise ValueError("batch_size must be a positive integer")
if log_every < 1:
    raise ValueError("log_every must be at least 1")
if max_steps is not None and max_steps < 1:
    raise ValueError("max_steps must be positive or None")

dec.train()
optimizer = torch.optim.AdamW(
    parameter for parameter in dec.parameters() if parameter.requires_grad
)
train_ds = ds["train"]
num_batches = (train_ds.num_rows + batch_size - 1) // batch_size
num_steps = num_batches if max_steps is None else min(max_steps, num_batches)
running_loss = torch.zeros((), device=device)
rows_since_log = 0

for step in range(num_steps):
    row_start = step * batch_size
    row_end = min(row_start + batch_size, train_ds.num_rows)
    rows_in_batch = row_end - row_start
    # Tokenization, padding, and target validation stay on CPU.
    inputs, target, row_spans = prepare_dataset_batch(
        train_ds, range(row_start, row_end), tokenizer
    )
    target = target.to(device)
    inputs = {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in inputs.items()
    }

    optimizer.zero_grad(set_to_none=True)
    logits = dec(inputs)
    noise = torch.randn((G, *logits.shape), device=logits.device, dtype=logits.dtype)
    candidates = logits.unsqueeze(0) + sigma * noise
    loss = batch_candidate_loss(
        candidates, target, inputs["qtype"], inputs["marker_mask"], row_spans, sigma
    )
    loss.backward()
    optimizer.step()
    # Release head gradients immediately so they do not pin MPS allocator blocks.
    optimizer.zero_grad(set_to_none=True)

    running_loss += loss.detach() * rows_in_batch
    rows_since_log += rows_in_batch
    if (step + 1) % log_every == 0 or step + 1 == num_steps:
        # Weight logs by row count so a partial batch is reported correctly.
        mean_loss = (running_loss / rows_since_log).item()
        print(
            f"step {step + 1}/{num_steps} | rows {row_end}/{train_ds.num_rows} "
            f"| loss {mean_loss:.4f}"
        )
        running_loss.zero_()
        rows_since_log = 0


step 1/19 | rows 16/300 | loss -0.0356


KeyboardInterrupt: 

In [ ]:
def boolq_to_schema():
    boolq = load_dataset("google/boolq")
    